# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates idempotent UPSERT/DELETE logic on a simulated stateful Iceberg table using DuckDB.

In [ ]:
import duckdb

con = duckdb.connect(database=':memory:')
print("DuckDB in-memory connection established.")

In [ ]:
# Create Target Table (Simulating Silver State)
con.execute("""
CREATE TABLE target_patients (
    patient_id VARCHAR PRIMARY KEY,
    status VARCHAR,
    updated_at TIMESTAMP
)
""")

# Insert Initial State
con.execute("""
INSERT INTO target_patients VALUES
('P1', 'ACTIVE', '2023-10-01 10:00:00'),
('P2', 'ACTIVE', '2023-10-01 10:00:00')
""")

print("Initial Target State:")
print(con.execute("SELECT * FROM target_patients").df())

In [ ]:
# Create Source View (Simulating incoming CDC Batch)
con.execute("""
CREATE VIEW cdc_batch AS SELECT * FROM (
    VALUES
    -- Late arriving, older update (should be ignored)
    ('P1', 'u', 'INACTIVE', '2023-10-01 09:00:00'::TIMESTAMP),
    -- Valid newer update
    ('P2', 'u', 'DISCHARGED', '2023-10-01 11:00:00'::TIMESTAMP),
    -- New Insert
    ('P3', 'c', 'ACTIVE', '2023-10-01 11:30:00'::TIMESTAMP),
    -- Delete operation
    ('P1', 'd', 'DELETED', '2023-10-01 12:00:00'::TIMESTAMP)
) AS t(patient_id, cdc_op, status, updated_at);
""")

print("\nIncoming CDC Batch:")
print(con.execute("SELECT * FROM cdc_batch ORDER BY updated_at").df())

In [ ]:
# Execute UPSERT/DELETE logic (Simulating Iceberg MERGE)
# Note: DuckDB doesn't have identical MERGE syntax to Spark SQL, so we simulate using Insert/On Conflict for UPSERT
# and a separate DELETE for simplicity in this notebook.

# 1. Handle Deletes
con.execute("""
DELETE FROM target_patients
WHERE patient_id IN (
    SELECT patient_id FROM cdc_batch
    WHERE cdc_op = 'd'
    AND updated_at > target_patients.updated_at
)
""")

# 2. Handle Upserts
con.execute("""
INSERT INTO target_patients (patient_id, status, updated_at)
SELECT patient_id, status, updated_at FROM cdc_batch WHERE cdc_op != 'd'
ON CONFLICT (patient_id) DO UPDATE SET
    status = CASE 
        WHEN EXCLUDED.updated_at > target_patients.updated_at THEN EXCLUDED.status
        ELSE target_patients.status 
    END,
    updated_at = CASE 
        WHEN EXCLUDED.updated_at > target_patients.updated_at THEN EXCLUDED.updated_at
        ELSE target_patients.updated_at 
    END
""")

print("\nFinal Target State After Idempotent Upsert/Delete:")
print(con.execute("SELECT * FROM target_patients ORDER BY patient_id").df())